# 👑 System One Apex V34: Dual-Titan MoS (Yi-34B L30 + Mistral Large 123B L44) + Dual-Scale Bessel Twin + Woodbury Bayesian SCache O(1)
## Name That LLM — Pure Proprietary System One Architecture (Zero 3rd-Party LoRAs, Zero VRAM Leak, Multi-Regime Calibrated)
---
### Architectural Stack:
1. **Statistical & Structural Anchors** (*explicitly declared per `AGENTS.md` / `SKILL.md`*):
   - `svc_open_pool`: Checksum-pinned 4-view Open-Pool `LinearSVC` anchor.
   - `svc_host_exact` & `svc_proxy_clean_8view`: Live-fitted Unicode-Typography + Lexical-Distortion `LinearSVC` anchors with multiplicity-gated Bessel prompt centering.
   - `deberta_v3_large`: Bidirectional syntactic encoder anchor.
2. **Compass 1 — 120D Host-Aware Forensic Frontier Scouts (`extract_forensic_compass1_120d`)**:
   - Exact Unicode typography ratios, Markdown geometry, sentence burstiness, UK/US spelling & Ghost-Jump discourse transitions.
3. **Compass 2 — Size-Capped Recursive Topic Clustering + Dual-Scale Bessel Centering (`m ∈ {1..18}`)**:
   - Exact Bessel degrees-of-freedom scaling $\sqrt{m / (m - 1)}$ + normalized multiplicity gate (`gate(1)=0.0`, `gate(18)=1.0`) + specialized `centered_arena` (`m ∈ {2,3,4}`) vs `centered` (`m=18`) heads.
4. **Compass 3 — Zero-Leak Multi-GPU FP16 `Mixture of Scars (MoS)` Layer-Streaming Engine**:
   - **Titan 2 (`Yi-34B` Golden-Slice L30, `7,288D` Co-Trained `SCache`)** & **Titan 3 (`Mistral Large 2 123B` Golden-Slice L44, `12,408D` Co-Trained `SCache`)** (`< 4.2 GiB` peak VRAM on T4; auto-detects **Titan 4 `Qwen3-Embedding` 8-Depth MoS Scar Experts** when attached).
5. **Intuition & Collective Memory (`UniversalS1ReflexiveMLP` 625D + `SCache O(1)` Exact Pair/Triple Woodbury & Cavity Bayesian Head + Cavity-EM + `ReflexDecision` S1 $\to$ S2 Clan Verifiers + Size-Scaled Soft Sinkhorn)**.


In [ ]:
import gc, glob, json, math, os, re, shutil, time
from collections import Counter
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open
from safetensors.torch import load_file

torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
L2I = {l: i for i, l in enumerate(LABELS)}
K = len(LABELS)
CLANS = {"openai": [0, 1, 2, 3, 4], "gemini": [5, 6, 7, 8], "claude": [9, 10, 11, 12, 13], "glm": [15, 16]}

def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def vote(name, P, w, rows=slice(None)):
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1.0))
    w_mean = float(np.mean(w))
    print(f"{name}: mean_weight {w_mean:.4f}, done after {time.time() - T0:.1f}s", flush=True)

def bessel_group_center(X, groups, mask=None):
    Xc = np.zeros_like(X, dtype=np.float32)
    ugs = np.unique(groups) if mask is None else np.unique(groups[mask])
    for g in ugs:
        m = (groups == g)
        cnt = int(m.sum())
        if cnt > 1:
            bessel = float(np.sqrt(cnt / float(cnt - 1)))
            Xc[m] = (X[m] - X[m].mean(axis=0, keepdims=True)) * bessel
    return Xc

def compute_normalized_twin_gate(sizes, kappa=2.5):
    s = np.asarray(sizes, dtype=np.float64)
    gate = np.zeros_like(s, dtype=np.float64)
    active = s >= 2
    dm = s[active] - 1.0
    kp = max(float(kappa), 1e-4)
    gate[active] = np.minimum(1.0, ((17.0 + kp) / 17.0) * (dm / (dm + kp)))
    return gate

class TestTimeSCache:
    """O(1) Associative Scar-Cache resonance over constant 32x32 scar covariance slots."""
    def __init__(self, slot_dim=32, resonance_scale=0.15):
        self.slot_dim = slot_dim
        self.resonance_scale = resonance_scale
        self.proj = None

    def apply_scache_resonance(self, X, groups, twin_mask=None):
        if self.proj is None:
            rng = np.random.RandomState(42)
            Q, _ = np.linalg.qr(rng.randn(X.shape[1], self.slot_dim))
            self.proj = Q.astype(np.float32)
        Z = X @ self.proj
        X_out = X.copy()
        ugs = np.unique(groups) if twin_mask is None else np.unique(groups[twin_mask])
        for g in ugs:
            m = (groups == g)
            cnt = int(m.sum())
            if cnt > 1:
                gate_m = min(1.0, (19.0 / 17.0) * (cnt - 1.0) / (cnt - 1.0 + 2.0))
                dz = Z[m] - Z[m].mean(axis=0, keepdims=True)
                W_scar = (dz.T @ dz) / float(cnt)
                X_out[m] += (self.resonance_scale * gate_m) * ((dz @ W_scar.T) @ self.proj.T)
        return X_out

"""
System One — Compass 1: 120D Host-Aware Forensic Frontier Scouts (Bússola 1)
============================================================================
Extrai determinísticamente 120 assinaturas criptográficas de tipografia Unicode,
geometria estrutural Markdown, explosividade (burstiness) rítmica de sentenças
e transições retóricas Ghost-Jump calibradas para o gerador oficial da competição.
"""


import math
import re
from collections import Counter
from typing import Iterable, List

import numpy as np

_WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['\u2019][A-Za-z0-9]+)*")
_SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
_NUM_LIST_RE = re.compile(r"(?:^|\n)\s*\d+[\.\)]\s")
_BUL_LIST_RE = re.compile(r"(?:^|\n)\s*[-*•]\s")
_HDR_RE = re.compile(r"(?:^|\n)#{1,6}\s")
_ITALIC_SINGLE_RE = re.compile(r"(?<!\*)\*(?!\*|\s)[^*\n]{1,80}(?<!\s|\*)\*(?!\*)")
_UNSPACED_EMDASH_RE = re.compile(r"\w\u2014\w")
_SPACED_EMDASH_RE = re.compile(r"\s\u2014\s")
_UK_SPELLINGS = frozenset(
    "colour colours coloured behaviour behaviours centre centres metre metres "
    "kilometre kilometres centimetre centimetres millimetre millimetres litre litres "
    "fibre fibres fibreglass tyre tyres analyse analyses analysed organise organises "
    "organised recognise recognises recognised realise realises realised优化 "
    "optimise optimises optimisedグレー grey honour honours honoured favour favours "
    "favoured neighbour neighbours دفاع defence offence license practise travelled "
    "travelling cancelled cancelling labelled labelling whilst amongst towards".split()
)
_US_SPELLINGS = frozenset(
    "color colors colored behavior behaviors center centers meter meters "
    "kilometer kilometers centimeter centimeters millimeter millimeters liter liters "
    "fiber fibers fiberglass tire tires analyze analyzes analyzed organize organizes "
    "organized recognize recognizes recognized realize realizes realized "
    "optimize optimizes optimized gray honor honors honored favor favors "
    "favored neighbor neighbors defense offense traveled traveling canceled "
    "canceling labeled labeling toward".split()
)

_GHOST_PHRASES = [
    "however,",
    "moreover,",
    "furthermore,",
    "therefore,",
    "thus,",
    "instead,",
    "rather,",
    "in contrast,",
    "on the other hand",
    "for example,",
    "for instance,",
    "specifically,",
    "in particular,",
    "overall,",
    "in short,",
    "in summary,",
    "in conclusion",
    "ultimately,",
    "first,",
    "second,",
    "finally,",
    "imagine ",
    "think of ",
    "consider ",
    "picture ",
    "at its core",
    "in essence",
    "simply put",
    "in other words",
    "here is ",
    "here's ",
    "let's ",
    "to understand",
    "that means",
    "this means",
    "because of this",
    "as a result",
    "e.g.",
    "i.e.",
]

_OPENING_STARTERS = (
    "when ",
    "while ",
    "although ",
    "if ",
    "imagine ",
    "think ",
    "in ",
    "the ",
    "a ",
    "to ",
)


def extract_forensic_compass1_120d(texts: Iterable[str]) -> np.ndarray:
    """
    Retorna matriz (N, 120) float32 com assinaturas forenses livres de NaN/Inf.
    """
    rows: List[List[float]] = []
    for raw in texts:
        t = "" if raw is None else str(raw)
        n_chars = len(t)
        nc = float(max(n_chars, 1))

        words_raw = _WORD_RE.findall(t)
        nw_int = len(words_raw)
        nw = float(max(nw_int, 1))
        h100 = 100.0 / nw

        # Normaliza apóstrofos curvos para análise lexical sem perder contagem Unicode bruta
        words_low = [w.lower().replace("\u2019", "'") for w in words_raw]

        # ------------------------------------------------------------------
        # 1. Tipografia Criptográfica Unicode & Pontuação Fina (32 dims)
        # ------------------------------------------------------------------
        c_ldq = t.count("\u201c")
        c_rdq = t.count("\u201d")
        c_curly_dq = c_ldq + c_rdq
        c_str_dq = t.count('"')
        tot_dq = c_curly_dq + c_str_dq

        c_lsq = t.count("\u2018")
        c_rsq = t.count("\u2019")
        c_curly_sq = c_lsq + c_rsq
        c_str_sq = t.count("'")
        tot_sq = c_curly_sq + c_str_sq

        c_em = t.count("\u2014")
        c_em_unspaced = len(_UNSPACED_EMDASH_RE.findall(t))
        c_em_spaced = len(_SPACED_EMDASH_RE.findall(t))
        c_en = t.count("\u2013")
        c_hyp = t.count("-")
        c_dhyp = t.count("--")
        c_shyp = t.count(" - ")
        c_ell_uni = t.count("\u2026")
        c_ell_asc = t.count("...")
        c_semi = t.count(";")
        c_colon = t.count(":")
        c_comma = t.count(",")
        c_excl = t.count("!")
        c_quest = t.count("?")
        c_slash = t.count("/")
        c_amp = t.count("&")
        c_pct = t.count("%")
        c_lparen = t.count("(")
        c_lbrack = t.count("[")
        non_ascii = sum(1 for ch in t if ord(ch) > 127)

        f_typo = [
            1.0 if c_curly_dq > 0 else 0.0,
            c_curly_dq * h100,
            c_str_dq * h100,
            c_curly_dq / (tot_dq + 1e-6) if tot_dq > 0 else 0.0,
            1.0 if tot_dq > 0 else 0.0,
            1.0 if c_curly_sq > 0 else 0.0,
            c_curly_sq * h100,
            c_str_sq * h100,
            c_curly_sq / (tot_sq + 1e-6) if tot_sq > 0 else 0.0,
            1.0 if tot_sq > 0 else 0.0,
            1.0 if c_em > 0 else 0.0,
            c_em * h100,
            c_em_unspaced * h100,
            c_em_spaced * h100,
            c_em_unspaced / (c_em + 1e-6) if c_em > 0 else 0.0,
            1.0 if c_en > 0 else 0.0,
            c_en * h100,
            c_hyp * h100,
            c_dhyp * h100,
            c_shyp * h100,
            (c_ell_uni + c_ell_asc) * h100,
            1.0 if c_semi > 0 else 0.0,
            c_semi * h100,
            1.0 if c_colon > 0 else 0.0,
            c_colon * h100,
            c_comma * h100,
            c_excl * h100,
            c_quest * h100,
            c_slash * h100,
            (c_amp + c_pct) * h100,
            c_lparen * h100,
            non_ascii / nc,
        ]  # 32

        # ------------------------------------------------------------------
        # 2. Geometria Estrutural Markdown & Parágrafos (20 dims)
        # ------------------------------------------------------------------
        c_bold = t.count("**") // 2
        c_ital = len(_ITALIC_SINGLE_RE.findall(t))
        c_code_fence = t.count("```") // 2
        c_backtick = max(0, t.count("`") - 6 * c_code_fence)
        c_hdr = len(_HDR_RE.findall(t))
        c_nlist = len(_NUM_LIST_RE.findall(t))
        c_blist = len(_BUL_LIST_RE.findall(t))
        c_bquote = t.count("\n>")
        c_lbrace = c_lbrack + t.count("{")

        paras = [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
        n_paras = max(len(paras), 1)
        para_lens = [max(len(_WORD_RE.findall(p)), 1) for p in paras] if paras else [nw_int]
        c_nl = t.count("\n")
        c_dnl = t.count("\n\n")

        f_struct = [
            1.0 if c_bold > 0 else 0.0,
            c_bold * h100,
            1.0 if c_ital > 0 else 0.0,
            c_ital * h100,
            c_backtick * h100,
            float(c_code_fence),
            float(c_hdr),
            1.0 if (c_nlist + c_blist) > 0 else 0.0,
            c_nlist * h100,
            c_blist * h100,
            float(c_bquote),
            c_lbrace * h100,
            float(n_paras),
            1.0 if n_paras == 1 else 0.0,
            float(np.mean(para_lens)) / 100.0,
            float(np.std(para_lens)) / 100.0 if n_paras > 1 else 0.0,
            para_lens[0] / nw,
            para_lens[-1] / nw,
            c_nl * h100,
            c_dnl * h100,
        ]  # 20 (cum 52)

        # ------------------------------------------------------------------
        # 3. Ritmo de Sentenças, Burstiness & Riqueza Lexical (24 dims)
        # ------------------------------------------------------------------
        sents = [s.strip() for s in _SENT_SPLIT_RE.split(t) if s.strip()]
        n_sents = max(len(sents), 1)
        sent_lens = [max(len(_WORD_RE.findall(s)), 1) for s in sents] if sents else [nw_int]
        s_arr = np.asarray(sent_lens, dtype=np.float32)
        s_mean = float(s_arr.mean())
        s_std = float(s_arr.std()) if len(s_arr) > 1 else 0.0
        s_med = float(np.median(s_arr))
        s_min = float(s_arr.min())
        s_max = float(s_arr.max())

        w_lens = np.asarray([len(w) for w in words_raw], dtype=np.float32) if nw_int > 0 else np.zeros(1, dtype=np.float32)
        counts = Counter(words_low)
        n_uniq = len(counts)
        hapax = sum(1 for v in counts.values() if v == 1)
        m2 = sum(v * v for v in counts.values())
        yule_k = 1e4 * (m2 - nw) / (nw * nw + 1e-6) if nw_int > 1 else 0.0

        upper_chars = sum(1 for ch in t if ch.isupper())
        digit_chars = sum(1 for ch in t if ch.isdigit())
        title_words = sum(1 for w in words_raw if w[0].isupper())
        allcaps_words = sum(1 for w in words_raw if len(w) >= 2 and w.isupper())
        num_tokens = sum(1 for w in words_raw if any(ch.isdigit() for ch in w))

        f_rhythm = [
            nw / 100.0,
            math.log1p(nw_int),
            nc / 500.0,
            float(w_lens.mean()) / 10.0,
            float(w_lens.std()) / 10.0,
            float(np.percentile(w_lens, 90)) / 10.0,
            float((w_lens >= 8).mean()),
            float((w_lens <= 3).mean()),
            n_uniq / nw,
            hapax / nw,
            min(yule_k / 200.0, 5.0),
            upper_chars / nc,
            digit_chars / nc,
            title_words / nw,
            allcaps_words * h100,
            num_tokens * h100,
            float(n_sents) / 10.0,
            s_mean / 25.0,
            s_std / 15.0,
            s_med / 25.0,
            s_min / 25.0,
            s_max / 40.0,
            s_std / (s_mean + 1e-6),
            c_comma / float(n_sents),
        ]  # 24 (cum 76)

        # ------------------------------------------------------------------
        # 4. Batedores Ghost-Jump: Contrações, UK/US & Discurso (44 dims)
        # ------------------------------------------------------------------
        t_low = t.lower().replace("\u2019", "'")
        n_nt = sum(1 for w in words_low if w.endswith("n't"))
        n_re_ve_ll = sum(1 for w in words_low if w.endswith(("'re", "'ve", "'ll", "'d")))
        n_its_contr = words_low.count("it's") + words_low.count("that's") + words_low.count("there's")
        n_it_is = t_low.count("it is ") + t_low.count("that is ") + t_low.count("do not ") + t_low.count("does not ") + t_low.count("cannot ")
        uk_hits = sum(1 for w in words_low if w in _UK_SPELLINGS)
        us_hits = sum(1 for w in words_low if w in _US_SPELLINGS)

        t_strip_low = t_low.lstrip()
        starts_question = 1.0 if (sents and sents[0].endswith("?")) else 0.0
        ends_exclamation = 1.0 if t.rstrip().endswith("!") else 0.0
        first_starter = 1.0 if t_strip_low.startswith(_OPENING_STARTERS[:5]) else 0.0

        f_ghost = [
            (n_nt + n_re_ve_ll + n_its_contr) * h100,
            n_nt * h100,
            n_it_is * h100,
            (n_nt + n_its_contr) / (n_nt + n_its_contr + n_it_is + 1e-6),
            uk_hits * h100,
            us_hits * h100,
            starts_question,
            ends_exclamation,
            first_starter,
        ]  # 9 -> remaining 35 from _GHOST_PHRASES[:35]

        for ph in _GHOST_PHRASES[:35]:
            f_ghost.append(t_low.count(ph) * h100)

        vec = f_typo + f_struct + f_rhythm + f_ghost
        assert len(vec) == 120, f"Expected 120 features, got {len(vec)}"
        rows.append(vec)

    out = np.asarray(rows, dtype=np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=5.0, neginf=-5.0)
"""
System One — Unified Zero-Leak Pure-PyTorch FP16 MoS Layer-Streaming Engine
===========================================================================
Suporta:
1. Qualquer backbone NF4 MoS (Mistral Large 123B, Yi-34B, Qwen3-Embedding-8B, Qwen-72B)
   com detecção automática de prefixo ('model.' vs ''), GQA, RoPE theta e left/right padding.
2. Isolamento estrito de escopo em _step_attn() e _step_mlp(): zero vazamento de tensores
   entre subcamadas (pico de VRAM <= 4.2 GiB por GPU T4 mesmo para N = 10.000 passagens).
3. Extração Multi-Depth simultânea opcional (ex: 8 fatias [4,8,12,16,20,24,30,36]) com
   custo zero de I/O extra + fusão in-place opcional de MoS Residual Scar Experts (dW = B @ A).
4. Janelas deslizantes de cobertura completa (covering windows sem truncamento) com agregação
   ponderada por contagem de tokens válidos.
"""


import gc
import json
import os
import time
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from safetensors import safe_open


def build_covering_windows(
    ids_list: Sequence[Sequence[int]],
    max_len: int = 512,
    stride: int = 384,
) -> Tuple[List[List[int]], np.ndarray, np.ndarray]:
    """
    Decompõe sequências longas em janelas deslizantes de até `max_len` tokens sem
    descartar parágrafos finais. Para sequências <= max_len, gera exatamente 1 janela.
    Retorna:
      win_ids: lista de sequências de tokens por janela
      doc_idx: índice do documento original de cada janela (M,)
      win_weights: peso proporcional ao comprimento útil de cada janela (M,)
    """
    win_ids: List[List[int]] = []
    doc_idx: List[int] = []
    win_weights: List[float] = []

    for i, seq in enumerate(ids_list):
        s = list(seq)
        L = len(s)
        if L <= max_len:
            win_ids.append(s)
            doc_idx.append(i)
            win_weights.append(float(max(L, 1)))
        else:
            starts = list(range(0, L - max_len + 1, stride))
            if starts[-1] + max_len < L:
                starts.append(L - max_len)
            for st in starts:
                chunk = s[st : st + max_len]
                win_ids.append(chunk)
                doc_idx.append(i)
                win_weights.append(float(len(chunk)))

    return win_ids, np.asarray(doc_idx, dtype=np.int64), np.asarray(win_weights, dtype=np.float32)


def aggregate_window_embeddings(
    Z_win: np.ndarray,
    doc_idx: np.ndarray,
    win_weights: np.ndarray,
    n_docs: int,
    unit_rms: bool = True,
) -> np.ndarray:
    """Agrega embeddings de janelas deslizantes de volta para (n_docs, D)."""
    D = Z_win.shape[1]
    out = np.zeros((n_docs, D), dtype=np.float32)
    w_sum = np.zeros((n_docs, 1), dtype=np.float32)
    np.add.at(out, doc_idx, Z_win * win_weights[:, None])
    np.add.at(w_sum[:, 0], doc_idx, win_weights)
    out /= np.maximum(w_sum, 1e-6)
    if unit_rms:
        rms = 1.0 / np.sqrt(np.mean(out * out, axis=-1, keepdims=True) + 1e-6)
        out = out * rms
    return out


def extract_mos_nf4_streaming(
    backbone_dir: str,
    ids_list: Sequence[Sequence[int]],
    pad_id: int,
    devices: Sequence[str],
    padding_side: str = "left",
    tag: str = "MoS",
    micro_bsz: int = 8,
    max_tokens_per_dev: int = 60000,
    capture_layers: Optional[Sequence[int]] = None,
    scar_tensors: Optional[Dict[str, torch.Tensor]] = None,
    scar_scale: float = 1.0,
    log_every: int = 10,
) -> np.ndarray:
    """
    Motor canônico puro-PyTorch FP16 de Layer-Streaming MoS.
    - Se `capture_layers` for None: retorna (N, d_model) da última camada do backbone.
    - Se `capture_layers` for uma lista de índices 1-based (ex: [4,8,12,16,20,24,30,36]):
      retorna (N, len(capture_layers) * d_model) concatenando os pools normalizados!
    """
    devices = list(dict.fromkeys(devices))
    with open(os.path.join(backbone_dir, "config.json"), "r") as f:
        cfg_raw = json.load(f)

    n_layers = int(cfg_raw["num_hidden_layers"])
    d_model = int(cfg_raw["hidden_size"])
    n_heads = int(cfg_raw["num_attention_heads"])
    n_kv = int(cfg_raw["num_key_value_heads"])
    head_dim = int(cfg_raw.get("head_dim") or (d_model // n_heads))
    eps = float(cfg_raw.get("rms_norm_eps", 1e-5))
    rope_params = cfg_raw.get("rope_parameters") or {}
    theta = float(cfg_raw.get("rope_theta") or rope_params.get("rope_theta") or 1000000.0)
    gqa_rep = n_heads // n_kv
    has_qkv_bias = bool(cfg_raw.get("attention_bias", False))

    index_path = os.path.join(backbone_dir, "model.safetensors.index.json")
    if os.path.exists(index_path):
        with open(index_path, "r") as f:
            idx_map = json.load(f)["weight_map"]
    else:
        single_st = os.path.join(backbone_dir, "model.safetensors")
        with safe_open(single_st, framework="pt", device="cpu") as f_single:
            idx_map = {k: "model.safetensors" for k in f_single.keys()}

    base_pfx = "model." if "model.embed_tokens.weight" in idx_map else ""
    layers_pfx = f"{base_pfx}layers."
    norm_key = f"{base_pfx}norm.weight"
    open_shards: Dict[str, object] = {}

    target_layers_1b = list(capture_layers) if capture_layers is not None else [n_layers]
    max_target_layer = max(target_layers_1b)
    if max_target_layer > n_layers:
        raise ValueError(f"Requested layer {max_target_layer} exceeds backbone num_hidden_layers={n_layers}")
    capture_map = {l_1b - 1: slot_i for slot_i, l_1b in enumerate(target_layers_1b)}

    def get_tensor_cpu(key: str) -> torch.Tensor:
        shard_name = idx_map[key]
        if shard_name not in open_shards:
            open_shards[shard_name] = safe_open(
                os.path.join(backbone_dir, shard_name), framework="pt", device="cpu"
            )
        return open_shards[shard_name].get_tensor(key)

    def maybe_get_bias(prefix: str, dev: str, dtype=torch.float16) -> Optional[torch.Tensor]:
        b_key = f"{prefix}.bias"
        if b_key in idx_map:
            return get_tensor_cpu(b_key).to(dev, dtype=dtype)
        return None

    def dequant_nf4_mos(prefix: str, dev: str, dtype=torch.float16) -> torch.Tensor:
        w_key = f"{prefix}.weight"
        q_key = f"{prefix}.weight.quant_state.bitsandbytes__nf4"
        if q_key not in idx_map:
            w_out = get_tensor_cpu(w_key).to(dev, dtype=dtype)
        else:
            w_u8 = get_tensor_cpu(w_key)
            abs_u8 = get_tensor_cpu(f"{prefix}.weight.absmax")
            nabs = get_tensor_cpu(f"{prefix}.weight.nested_absmax")
            qmap = get_tensor_cpu(f"{prefix}.weight.quant_map")
            nqmap = get_tensor_cpu(f"{prefix}.weight.nested_quant_map")
            qs_b = get_tensor_cpu(q_key)
            qs = json.loads(bytes(qs_b.numpy()).decode("utf-8"))
            shape = tuple(qs["shape"])
            bs = int(qs["blocksize"])
            nbs = int(qs["nested_blocksize"])
            noff = float(qs["nested_offset"])

            qmap_d = qmap.to(dev, dtype=torch.float32)
            nqmap_d = nqmap.to(dev, dtype=torch.float32)
            real_abs = (
                nqmap_d[abs_u8.to(dev).long()].view(-1, nbs)
                * nabs.to(dev, dtype=torch.float32).view(-1, 1)
            ).reshape(-1) + noff
            idx256 = torch.arange(256, device=dev, dtype=torch.long)
            pair_lut = torch.stack([qmap_d[idx256 >> 4], qmap_d[idx256 & 0x0F]], dim=1)

            w_flat = w_u8.reshape(-1)
            out_flat = torch.empty(shape[0] * shape[1], device=dev, dtype=dtype)
            chunk_bytes = 67108864  # 64 MB packed = 128M weights (keeps LUT temp buffer <= 512 MB!)
            for b_s in range(0, w_flat.numel(), chunk_bytes):
                b_e = min(b_s + chunk_bytes, w_flat.numel())
                c_u8 = w_flat[b_s:b_e].to(dev, non_blocking=True)
                vals = pair_lut[c_u8.to(torch.int32)].view(-1, bs)
                vals.mul_(real_abs[(b_s * 2) // bs : (b_e * 2) // bs, None])
                out_flat[b_s * 2 : b_e * 2] = vals.view(-1).to(dtype)
                del c_u8, vals
            del qmap_d, nqmap_d, real_abs, idx256, pair_lut
            w_out = out_flat.view(shape)

        # In-place fusion of proprietary MoS Residual Scar Experts (dW = scar_scale * B @ A)
        if scar_tensors is not None:
            alt_pfx = prefix[len("model.") :] if prefix.startswith("model.") else f"model.{prefix}"
            a_key = f"{prefix}.scar_A" if f"{prefix}.scar_A" in scar_tensors else f"{alt_pfx}.scar_A"
            b_key = f"{prefix}.scar_B" if f"{prefix}.scar_B" in scar_tensors else f"{alt_pfx}.scar_B"
            if a_key in scar_tensors and b_key in scar_tensors:
                sA = scar_tensors[a_key].to(dev, dtype=torch.float32)
                sB = scar_tensors[b_key].to(dev, dtype=torch.float32)
                w_out.add_((torch.matmul(sB, sA) * scar_scale).to(dtype))
                del sA, sB
        return w_out

    clean_ids = [list(seq) if len(seq) > 0 else [pad_id] for seq in ids_list]
    n_samples = len(clean_ids)
    order = np.argsort([len(x) for x in clean_ids], kind="stable")

    micro_specs = []
    for i in range(0, n_samples, micro_bsz):
        b_idx = order[i : i + micro_bsz]
        max_s = max(len(clean_ids[j]) for j in b_idx)
        inp_ids = torch.full((len(b_idx), max_s), pad_id, dtype=torch.long)
        att_msk = torch.zeros((len(b_idx), max_s), dtype=torch.bool)
        for r, j in enumerate(b_idx):
            seq = clean_ids[j]
            L = len(seq)
            if padding_side == "left":
                inp_ids[r, max_s - L :] = torch.tensor(seq, dtype=torch.long)
                att_msk[r, max_s - L :] = True
            else:
                inp_ids[r, :L] = torch.tensor(seq, dtype=torch.long)
                att_msk[r, :L] = True
        micro_specs.append((b_idx, inp_ids, att_msk))

    max_tokens_macro = max(int(max_tokens_per_dev) * max(len(devices), 1), 1)
    macro_chunks = []
    cur_chunk, cur_toks = [], 0
    for spec in micro_specs:
        t_cnt = int(spec[1].numel())
        if cur_chunk and cur_toks + t_cnt > max_tokens_macro:
            macro_chunks.append(cur_chunk)
            cur_chunk, cur_toks = [], 0
        cur_chunk.append(spec)
        cur_toks += t_cnt
    if cur_chunk:
        macro_chunks.append(cur_chunk)

    n_slots = len(target_layers_1b)
    Z_out = np.zeros((n_samples, n_slots * d_model), dtype=np.float32)

    def rot_half(z: torch.Tensor) -> torch.Tensor:
        z1, z2 = z[..., : head_dim // 2], z[..., head_dim // 2 :]
        return torch.cat([-z2, z1], dim=-1)

    def build_rope_and_bias(att_msk_d: torch.Tensor, dev: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        bsz, S = att_msk_d.shape
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32, device=dev) / head_dim))
        t = torch.arange(S, dtype=torch.float32, device=dev)
        freqs = torch.outer(t, inv_freq)
        emb_pos = torch.cat([freqs, freqs], dim=-1)
        cos = emb_pos.cos().to(torch.float16)[None, None, :, :]
        sin = emb_pos.sin().to(torch.float16)[None, None, :, :]

        causal = torch.tril(torch.ones((S, S), dtype=torch.bool, device=dev))
        allow = causal[None, None, :, :] & att_msk_d[:, None, None, :]
        all_masked = ~allow.any(dim=-1, keepdim=True)
        allow = allow | (all_masked & causal[None, None, :, :])
        attn_bias = torch.zeros((bsz, 1, S, S), dtype=torch.float16, device=dev).masked_fill(
            ~allow, torch.finfo(torch.float16).min
        )
        return cos, sin, attn_bias

    def _step_attn(lp: str, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        d0 = active_devs[0]
        in_ln_0 = get_tensor_cpu(f"{lp}.input_layernorm.weight").to(d0, dtype=torch.float16)
        q_0 = dequant_nf4_mos(f"{lp}.self_attn.q_proj", d0)
        k_0 = dequant_nf4_mos(f"{lp}.self_attn.k_proj", d0)
        v_0 = dequant_nf4_mos(f"{lp}.self_attn.v_proj", d0)
        o_0 = dequant_nf4_mos(f"{lp}.self_attn.o_proj", d0)
        qb_0 = maybe_get_bias(f"{lp}.self_attn.q_proj", d0)
        kb_0 = maybe_get_bias(f"{lp}.self_attn.k_proj", d0)
        vb_0 = maybe_get_bias(f"{lp}.self_attn.v_proj", d0)
        ob_0 = maybe_get_bias(f"{lp}.self_attn.o_proj", d0)
        qn_0 = get_tensor_cpu(f"{lp}.self_attn.q_norm.weight").to(d0, dtype=torch.float16) if f"{lp}.self_attn.q_norm.weight" in idx_map else None
        kn_0 = get_tensor_cpu(f"{lp}.self_attn.k_norm.weight").to(d0, dtype=torch.float16) if f"{lp}.self_attn.k_norm.weight" in idx_map else None

        attn_w = {d0: (in_ln_0, q_0, k_0, v_0, o_0, qb_0, kb_0, vb_0, ob_0, qn_0, kn_0)}
        for d in active_devs[1:]:
            attn_w[d] = tuple(
                t.to(d, non_blocking=True) if t is not None else None for t in attn_w[d0]
            )
        del in_ln_0, q_0, k_0, v_0, o_0, qb_0, kb_0, vb_0, ob_0, qn_0, kn_0

        max_b = max(len(dev_states[d]) for d in active_devs)
        for bi in range(max_b):
            for d in active_devs:
                if bi < len(dev_states[d]):
                    _, x, att_msk_d = dev_states[d][bi]
                    in_ln, qw, kw, vw, ow, qb, kb, vb, ob, qn, kn = attn_w[d]
                    bsz, S, _ = x.shape
                    cos, sin, attn_bias = build_rope_and_bias(att_msk_d, d)
                    xf = x.float()
                    normed = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * in_ln
                    del xf
                    q = F.linear(normed, qw, qb).view(bsz, S, n_heads, head_dim)
                    k = F.linear(normed, kw, kb).view(bsz, S, n_kv, head_dim)
                    v = F.linear(normed, vw, vb).view(bsz, S, n_kv, head_dim).transpose(1, 2)
                    del normed
                    if qn is not None:
                        qf = q.float()
                        q = (qf * torch.rsqrt(qf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * qn
                    if kn is not None:
                        kf = k.float()
                        k = (kf * torch.rsqrt(kf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * kn
                    q = q.transpose(1, 2)
                    k = k.transpose(1, 2)
                    q = (q * cos) + (rot_half(q) * sin)
                    k = (k * cos) + (rot_half(k) * sin)
                    k_rep = k.repeat_interleave(gqa_rep, dim=1) if gqa_rep > 1 else k
                    v_rep = v.repeat_interleave(gqa_rep, dim=1) if gqa_rep > 1 else v
                    ao = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=attn_bias)
                    dev_states[d][bi][1] = x + F.linear(
                        ao.transpose(1, 2).contiguous().view(bsz, S, n_heads * head_dim), ow, ob
                    )
                    del cos, sin, attn_bias, q, k, v, k_rep, v_rep, ao
        del attn_w

    def _step_mlp(lp: str, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        d0 = active_devs[0]
        post_ln_0 = get_tensor_cpu(f"{lp}.post_attention_layernorm.weight").to(d0, dtype=torch.float16)
        gate_0 = dequant_nf4_mos(f"{lp}.mlp.gate_proj", d0)
        up_0 = dequant_nf4_mos(f"{lp}.mlp.up_proj", d0)
        down_0 = dequant_nf4_mos(f"{lp}.mlp.down_proj", d0)

        mlp_w = {d0: (post_ln_0, gate_0, up_0, down_0)}
        for d in active_devs[1:]:
            mlp_w[d] = tuple(t.to(d, non_blocking=True) for t in mlp_w[d0])
        del post_ln_0, gate_0, up_0, down_0

        max_b = max(len(dev_states[d]) for d in active_devs)
        for bi in range(max_b):
            for d in active_devs:
                if bi < len(dev_states[d]):
                    x = dev_states[d][bi][1]
                    pln, gw, uw, dw = mlp_w[d]
                    xf = x.float()
                    normed = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * pln
                    del xf
                    act = F.silu(F.linear(normed, gw))
                    act.mul_(F.linear(normed, uw))
                    del normed
                    dev_states[d][bi][1] = x + F.linear(act, dw)
                    del act
        del mlp_w

    def _pool_slot(slot_i: int, apply_final_norm: bool, final_ln_cpu: torch.Tensor, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        col_s = slot_i * d_model
        col_e = col_s + d_model
        for d in active_devs:
            fln = final_ln_cpu.to(d, dtype=torch.float16) if apply_final_norm else None
            for b_idx, x, att_msk_d in dev_states[d]:
                xf = x.float()
                if fln is not None:
                    x_use = ((xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * fln).float()
                else:
                    x_use = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
                m = att_msk_d.unsqueeze(-1).float()
                pooled = (x_use * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
                rms = torch.rsqrt(pooled.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
                Z_out[b_idx, col_s:col_e] = (pooled * rms).cpu().numpy()
                del xf, x_use, m, pooled, rms
            del fln

    t_mos0 = time.time()
    with torch.no_grad():
        for mc_idx, m_specs in enumerate(macro_chunks):
            open_shards.clear()
            emb_w = get_tensor_cpu(f"{base_pfx}embed_tokens.weight")
            dev_states = {d: [] for d in devices}
            for bi, (b_idx, inp_ids, att_msk) in enumerate(m_specs):
                d = devices[bi % len(devices)]
                x = F.embedding(inp_ids, emb_w).to(d, dtype=torch.float16)
                dev_states[d].append([b_idx, x, att_msk.to(d)])
            del emb_w
            active_devs = [d for d in devices if len(dev_states[d]) > 0]
            final_ln_cpu = get_tensor_cpu(norm_key)

            for l in range(max_target_layer):
                lp = f"{base_pfx}layers.{l}"
                needed_from_l = {
                    idx_map[k]
                    for k in idx_map
                    if k == norm_key
                    or (k.startswith(layers_pfx) and int(k[len(layers_pfx) :].split(".")[0]) >= l)
                }
                for s_name in list(open_shards.keys()):
                    if s_name not in needed_from_l:
                        del open_shards[s_name]

                _step_attn(lp, active_devs, dev_states)
                _step_mlp(lp, active_devs, dev_states)

                if l in capture_map:
                    _pool_slot(
                        capture_map[l],
                        apply_final_norm=(l == n_layers - 1),
                        final_ln_cpu=final_ln_cpu,
                        active_devs=active_devs,
                        dev_states=dev_states,
                    )

                if (l + 1) % log_every == 0 or (l + 1) == max_target_layer:
                    gc.collect()
                    vram_gb = (
                        torch.cuda.memory_allocated(0) / (1024 ** 3)
                        if torch.cuda.is_available()
                        else 0.0
                    )
                    print(
                        f"  [MoS Stream - {tag}] Macro {mc_idx+1}/{len(macro_chunks)} | "
                        f"Layer {l+1}/{max_target_layer} | Elapsed {time.time()-t_mos0:.1f}s | "
                        f"GPU0 Post-Layer Alloc {vram_gb:.2f} GiB",
                        flush=True,
                    )

            del dev_states, final_ln_cpu
            open_shards.clear()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return Z_out


def extract_qwen72b_mos_l40_streaming(
    mos_dir: str,
    ids_list: Sequence[Sequence[int]],
    pad_id: int,
    devices: Sequence[str],
    num_groups: int = 5,
    scar_scale: float = 0.10,
    micro_bsz: int = 8,
    max_tokens_per_dev: int = 60000,
    tag: str = "Titan1-Qwen72B-L40",
) -> np.ndarray:
    """
    Motor Zero-Leak Multi-GPU FP16 para o Titan 1 (`Qwen 2.5 72B MoS Golden-Slice L40`):
    - Lê `mos_embeddings_and_head.safetensors` + `mos_group_0..4.safetensors` (INT4 Block-128
      Universal Shared MLP Base + Rank-64 BF16 Scar Experts).
    - Right-padding estrito + atenção causal exata (`is_causal=True`), garantindo invariância
      matemática ao tamanho do batch/padding e paridade exata (`cos > 0.9999`) com
      `features_70b_l40_train_full.npy`!
    - Pico de VRAM <= 2.8 GiB por GPU T4 (desaloca cada grupo antes de abrir o próximo).
    """
    devices = list(dict.fromkeys(devices))
    d_model = 8192
    n_heads = 64
    n_kv = 8
    head_dim = 128
    gqa_rep = 8
    inter_dim = 29568
    eps = 1e-6
    theta = 1000000.0
    bpt = d_model // 128  # 64 blocos INT4 por token

    emb_path = os.path.join(mos_dir, "mos_embeddings_and_head.safetensors")
    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"Missing {emb_path}")

    clean_ids = [list(seq) if len(seq) > 0 else [pad_id] for seq in ids_list]
    n_samples = len(clean_ids)
    order = np.argsort([len(x) for x in clean_ids], kind="stable")

    micro_specs = []
    for i in range(0, n_samples, micro_bsz):
        b_idx = order[i : i + micro_bsz]
        max_s = max(len(clean_ids[j]) for j in b_idx)
        inp_ids = torch.full((len(b_idx), max_s), pad_id, dtype=torch.long)
        att_msk = torch.zeros((len(b_idx), max_s), dtype=torch.bool)
        for r, j in enumerate(b_idx):
            seq = clean_ids[j]
            L = len(seq)
            inp_ids[r, :L] = torch.tensor(seq, dtype=torch.long)
            att_msk[r, :L] = True
        micro_specs.append((b_idx, inp_ids, att_msk))

    max_tokens_macro = max(int(max_tokens_per_dev) * max(len(devices), 1), 1)
    macro_chunks = []
    cur_chunk, cur_toks = [], 0
    for spec in micro_specs:
        t_cnt = int(spec[1].numel())
        if cur_chunk and cur_toks + t_cnt > max_tokens_macro:
            macro_chunks.append(cur_chunk)
            cur_chunk, cur_toks = [], 0
        cur_chunk.append(spec)
        cur_toks += t_cnt
    if cur_chunk:
        macro_chunks.append(cur_chunk)

    Z_out = np.zeros((n_samples, d_model), dtype=np.float32)

    def dequant_int4_block128(packed: torch.Tensor, scales: torch.Tensor, shape: tuple, dev: str) -> torch.Tensor:
        p = packed.to(dev, non_blocking=True)
        s = scales.to(dev, dtype=torch.float16, non_blocking=True)
        low = ((p & 0x0F).to(torch.int16) - 8).to(torch.float16)
        high = (((p >> 4) & 0x0F).to(torch.int16) - 8).to(torch.float16)
        unp = torch.empty((p.shape[0], p.shape[1] * 2), dtype=torch.float16, device=dev)
        unp[:, 0::2] = low
        unp[:, 1::2] = high
        del p, low, high
        unp.mul_(s)
        del s
        return unp.view(shape)

    def rot_half(z: torch.Tensor) -> torch.Tensor:
        z1, z2 = z[..., : head_dim // 2], z[..., head_dim // 2 :]
        return torch.cat([-z2, z1], dim=-1)

    def build_rope_72b(S: int, dev: str) -> Tuple[torch.Tensor, torch.Tensor]:
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32, device=dev) / head_dim))
        t = torch.arange(S, dtype=torch.float32, device=dev)
        freqs = torch.outer(t, inv_freq)
        emb_pos = torch.cat([freqs, freqs], dim=-1)
        return emb_pos.cos().to(torch.float16)[None, None, :, :], emb_pos.sin().to(torch.float16)[None, None, :, :]

    def _step_attn_72b(f_grp, grp_keys: set, l: int, active_devs: List[str], dev_states: Dict[str, list]) -> None:
        d0 = active_devs[0]
        lp = f"model.layers.{l}"
        in_ln_0 = f_grp.get_tensor(f"{lp}.input_layernorm.weight").to(d0, dtype=torch.float16)
        q_0 = dequant_int4_block128(f_grp.get_tensor(f"{lp}.self_attn.q_proj.weight"), f_grp.get_tensor(f"{lp}.self_attn.q_proj.weight.scales"), (d_model, d_model), d0)
        k_0 = dequant_int4_block128(f_grp.get_tensor(f"{lp}.self_attn.k_proj.weight"), f_grp.get_tensor(f"{lp}.self_attn.k_proj.weight.scales"), (n_kv * head_dim, d_model), d0)
        v_0 = dequant_int4_block128(f_grp.get_tensor(f"{lp}.self_attn.v_proj.weight"), f_grp.get_tensor(f"{lp}.self_attn.v_proj.weight.scales"), (n_kv * head_dim, d_model), d0)
        o_0 = dequant_int4_block128(f_grp.get_tensor(f"{lp}.self_attn.o_proj.weight"), f_grp.get_tensor(f"{lp}.self_attn.o_proj.weight.scales"), (d_model, d_model), d0)
        qb_0 = f_grp.get_tensor(f"{lp}.self_attn.q_proj.bias").to(d0, dtype=torch.float16) if f"{lp}.self_attn.q_proj.bias" in grp_keys else None
        kb_0 = f_grp.get_tensor(f"{lp}.self_attn.k_proj.bias").to(d0, dtype=torch.float16) if f"{lp}.self_attn.k_proj.bias" in grp_keys else None
        vb_0 = f_grp.get_tensor(f"{lp}.self_attn.v_proj.bias").to(d0, dtype=torch.float16) if f"{lp}.self_attn.v_proj.bias" in grp_keys else None

        attn_w = {d0: (in_ln_0, q_0, k_0, v_0, o_0, qb_0, kb_0, vb_0)}
        for d in active_devs[1:]:
            attn_w[d] = tuple(t.to(d, non_blocking=True) if t is not None else None for t in attn_w[d0])
        del in_ln_0, q_0, k_0, v_0, o_0, qb_0, kb_0, vb_0

        max_b = max(len(dev_states[d]) for d in active_devs)
        for bi in range(max_b):
            for d in active_devs:
                if bi < len(dev_states[d]):
                    _, x, _ = dev_states[d][bi]
                    in_ln, qw, kw, vw, ow, qb, kb, vb = attn_w[d]
                    bsz, S, _ = x.shape
                    cos, sin = build_rope_72b(S, d)
                    xf = x.float()
                    normed = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * in_ln
                    del xf
                    q = F.linear(normed, qw, qb).view(bsz, S, n_heads, head_dim).transpose(1, 2)
                    k = F.linear(normed, kw, kb).view(bsz, S, n_kv, head_dim).transpose(1, 2)
                    v = F.linear(normed, vw, vb).view(bsz, S, n_kv, head_dim).transpose(1, 2)
                    del normed
                    q = (q * cos) + (rot_half(q) * sin)
                    k = (k * cos) + (rot_half(k) * sin)
                    k_rep = k.repeat_interleave(gqa_rep, dim=1)
                    v_rep = v.repeat_interleave(gqa_rep, dim=1)
                    ao = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=(S > 1))
                    dev_states[d][bi][1] = x + F.linear(ao.transpose(1, 2).contiguous().view(bsz, S, d_model), ow)
                    del cos, sin, q, k, v, k_rep, v_rep, ao
        del attn_w

    def _step_mlp_72b(
        f_grp,
        l: int,
        start_l: int,
        base_mlp_w: Dict[str, Tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
        active_devs: List[str],
        dev_states: Dict[str, list],
    ) -> None:
        d0 = active_devs[0]
        lp = f"model.layers.{l}"
        pln_0 = f_grp.get_tensor(f"{lp}.post_attention_layernorm.weight").to(d0, dtype=torch.float16)
        if l > start_l:
            gA_0 = f_grp.get_tensor(f"{lp}.mlp.gate_proj.expert.A").to(d0, dtype=torch.float16)
            gB_0 = f_grp.get_tensor(f"{lp}.mlp.gate_proj.expert.B").to(d0, dtype=torch.float16)
            uA_0 = f_grp.get_tensor(f"{lp}.mlp.up_proj.expert.A").to(d0, dtype=torch.float16)
            uB_0 = f_grp.get_tensor(f"{lp}.mlp.up_proj.expert.B").to(d0, dtype=torch.float16)
            dA_0 = f_grp.get_tensor(f"{lp}.mlp.down_proj.expert.A").to(d0, dtype=torch.float16)
            dB_0 = f_grp.get_tensor(f"{lp}.mlp.down_proj.expert.B").to(d0, dtype=torch.float16)
            sc_0 = (pln_0, gA_0, gB_0, uA_0, uB_0, dA_0, dB_0)
            del pln_0, gA_0, gB_0, uA_0, uB_0, dA_0, dB_0
        else:
            sc_0 = (pln_0, None, None, None, None, None, None)
            del pln_0

        sc_w = {d0: sc_0}
        for d in active_devs[1:]:
            sc_w[d] = tuple(t.to(d, non_blocking=True) if t is not None else None for t in sc_0)
        del sc_0

        max_b = max(len(dev_states[d]) for d in active_devs)
        for bi in range(max_b):
            for d in active_devs:
                if bi < len(dev_states[d]):
                    x = dev_states[d][bi][1]
                    bg, bu, bd = base_mlp_w[d]
                    pln, gA, gB, uA, uB, dA, dB = sc_w[d]
                    xf = x.float()
                    normed = (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(torch.float16) * pln
                    del xf
                    if gA is None:
                        act = F.silu(F.linear(normed, bg))
                        act.mul_(F.linear(normed, bu))
                        del normed
                        mlp_out = F.linear(act, bd)
                        del act
                    else:
                        gate = F.linear(normed, bg)
                        gate.add_(((normed @ gA) @ gB), alpha=scar_scale)
                        gate = F.silu(gate)
                        up = F.linear(normed, bu)
                        up.add_(((normed @ uA) @ uB), alpha=scar_scale)
                        del normed
                        inter = gate.mul_(up)
                        del up
                        mlp_out = F.linear(inter, bd)
                        mlp_out.add_(((inter @ dA) @ dB), alpha=scar_scale)
                        del inter
                    dev_states[d][bi][1] = x + mlp_out
                    del mlp_out
        del sc_w

    t_mos0 = time.time()
    boff = torch.arange(bpt, dtype=torch.long)
    with torch.no_grad():
        for mc_idx, m_specs in enumerate(macro_chunks):
            with safe_open(emb_path, framework="pt", device="cpu") as f_emb:
                ek = "model.embed_tokens.weight" if "model.embed_tokens.weight" in f_emb.keys() else "embed_tokens.weight"
                emb_p = f_emb.get_tensor(ek)
                emb_s = f_emb.get_tensor(f"{ek}.scales")
                dev_states = {d: [] for d in devices}
                for bi, (b_idx, inp_ids, att_msk) in enumerate(m_specs):
                    d = devices[bi % len(devices)]
                    bsz, max_s = inp_ids.shape
                    flat_bidx = (inp_ids.unsqueeze(-1) * bpt + boff).reshape(-1)
                    x = dequant_int4_block128(emb_p[flat_bidx], emb_s[flat_bidx], (bsz, max_s, d_model), d)
                    dev_states[d].append([b_idx, x, att_msk.to(d)])
                del emb_p, emb_s

            active_devs = [d for d in devices if len(dev_states[d]) > 0]
            for g_idx in range(num_groups):
                gp = os.path.join(mos_dir, f"mos_group_{g_idx}.safetensors")
                with safe_open(gp, framework="pt", device="cpu") as f_grp:
                    grp_keys = set(f_grp.keys())
                    d0 = active_devs[0]
                    bg_0 = dequant_int4_block128(
                        f_grp.get_tensor(f"mos_base.group_{g_idx}.mlp.gate_proj.weight"),
                        f_grp.get_tensor(f"mos_base.group_{g_idx}.mlp.gate_proj.weight.scales"),
                        (inter_dim, d_model), d0,
                    )
                    bu_0 = dequant_int4_block128(
                        f_grp.get_tensor(f"mos_base.group_{g_idx}.mlp.up_proj.weight"),
                        f_grp.get_tensor(f"mos_base.group_{g_idx}.mlp.up_proj.weight.scales"),
                        (inter_dim, d_model), d0,
                    )
                    bd_0 = dequant_int4_block128(
                        f_grp.get_tensor(f"mos_base.group_{g_idx}.mlp.down_proj.weight"),
                        f_grp.get_tensor(f"mos_base.group_{g_idx}.mlp.down_proj.weight.scales"),
                        (d_model, inter_dim), d0,
                    )
                    base_mlp_w = {d0: (bg_0, bu_0, bd_0)}
                    for d in active_devs[1:]:
                        base_mlp_w[d] = tuple(t.to(d, non_blocking=True) for t in base_mlp_w[d0])
                    del bg_0, bu_0, bd_0

                    start_l = g_idx * 8
                    for l in range(start_l, start_l + 8):
                        _step_attn_72b(f_grp, grp_keys, l, active_devs, dev_states)
                        _step_mlp_72b(f_grp, l, start_l, base_mlp_w, active_devs, dev_states)

                    del base_mlp_w
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                vram_gb = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
                print(
                    f"  [MoS Stream - {tag}] Macro {mc_idx+1}/{len(macro_chunks)} | "
                    f"Group {g_idx+1}/{num_groups} (L{g_idx*8+8}/{num_groups*8}) | "
                    f"Elapsed {time.time()-t_mos0:.1f}s | GPU0 Post-Group Alloc {vram_gb:.2f} GiB",
                    flush=True,
                )

            for d in active_devs:
                for b_idx, x, att_msk_d in dev_states[d]:
                    m = att_msk_d.unsqueeze(-1).float()
                    pooled = (x.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
                    rms = torch.rsqrt(pooled.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
                    Z_out[b_idx] = (pooled * rms).cpu().numpy()
            del dev_states
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return Z_out
apex_meta = json.loads(Path(find("meta_milestone1_system_one_apex.json")).read_text())
assert apex_meta["labels"] == LABELS
apex_st = load_file(find("head_milestone1_system_one_apex.safetensors"))
LW = apex_meta["learned_weights"]
TEMPS = apex_meta.get("temperatures", {})

W_SVC = float(LW["c2_svc"])
W_HOST_SVC = float(LW["host_svc"])
W_PROXY_SVC = float(LW["proxy_svc"])
W_FORENSIC120 = float(LW["forensic120"])
W_DEBERTA = 0.28
W_T2_PLAIN = float(LW["t2_plain"])
W_T2_TWIN = float(LW["t2_twin"])
W_T3_PLAIN = float(LW["t3_plain"])
W_T3_TWIN = float(LW["t3_twin"])
W_S1_MLP = float(LW["s1_mlp"])
W_SCACHE_BAYES = float(LW["scache_bayes"])

T_SVC = float(TEMPS.get("svc", 0.85))
T_TITAN = float(TEMPS.get("titan", 4.2))
TWIN_KAPPA = float(apex_meta.get("best_twin_kappa", 2.5))
CAVITY_GAMMA = float(apex_meta["best_cavity_gamma"])
S2_BETA = float(apex_meta["best_s2_beta"])
SINKHORN_MIX = float(apex_meta["best_sinkhorn_target_mix"])
TWIN = dict(thresh=float(apex_meta.get("cluster_base_thresh", 0.89)), min_size=2, max_size=18)

train = pd.read_csv(find("train.csv"))
train["text"] = train["text"].fillna("").astype(str)
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map(L2I).values
LOGIT = np.zeros((len(test), K), dtype=np.float64)
print(f"train={train.shape} | test={test.shape} | gpus={torch.cuda.device_count()} | OOF_m18={apex_meta['oof_v33_apex_bal_acc']*100:.2f}%", flush=True)


In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS_C2 = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]

def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)

def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS_C2 or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))

SOURCE_CODE_SHA256 = "5864b360ef4b470726b6ae2018489e285651dfb2a305d24b1c6884666d81f800"
import hashlib, importlib.metadata, joblib
from pathlib import Path
EXPECTED_CACHE_SHA256 = "f294ce929714e852a5a3f1d34ff7a5c5f0a0df3d73bd4a24ae327cbde4d96352"
def _sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()
def _one(pattern):
    hits=sorted(set(glob.glob(f"{INPUT_DIR}/**/{pattern}",recursive=True)))
    if len(hits)!=1: raise RuntimeError(("Ambiguous or absent input",pattern,hits))
    return Path(hits[0])
mp = _one("ntl620_svc_prefit_v1/manifest.json")
manifest=json.loads(mp.read_text())
assert EXPECTED_CACHE_SHA256 != "SET_FROM_CPU_MANIFEST_BEFORE_GPU", "CPU cache checksum not pinned"
assert manifest["schema"]=="ntl620-svc-prefit-v1" and manifest["source_version"]==361311282
assert manifest["source_code_sha256"] == SOURCE_CODE_SHA256
assert manifest["labels"]==LABELS and manifest["views"]==[list(v) for v in VIEWS]
assert manifest["classes"]==list(range(K)) and not manifest["fit_used_test_rows"]
assert manifest["sample_cache_reload_exact"] is True
assert _sha(_one("train.csv"))==manifest["train_sha256"]
assert _sha(_one("pool_v4.parquet"))==manifest["pool_sha256"]
for package, expected in manifest["versions"].items():
    assert importlib.metadata.version(package)==expected, (package,expected,importlib.metadata.version(package))
cp=mp.parent/"svc_prefit.joblib"
assert manifest["cache_sha256"]==EXPECTED_CACHE_SHA256 and _sha(cp)==EXPECTED_CACHE_SHA256
cache=joblib.load(cp)  # checksum-pinned statistical anchor
assert cache["labels"]==LABELS and cache["svc"].classes_.tolist()==list(range(K))
B=[]
for vec,(view,lo,hi,max_features,weight) in zip(cache["vectorizers"],VIEWS):
    prep=shape_text if view=="shape" else small_words_text if view=="small" else (lambda s:s)
    B.append(vec.transform([prep(t) for t in test.text])*weight)
assert len(cache["vectorizers"])==len(VIEWS)==4
vote("svc_open_pool", softmax(cache["svc"].decision_function(hstack(B,format="csr")) / 0.85), W_SVC)
del B, cache
gc.collect()


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
    S_mat = (Xt @ Xt.T).toarray().astype(np.float32)
    del Xt
    D = np.clip(1.0 - S_mat, 0.0, None)
    np.fill_diagonal(D, 0.0)
    Z_lk = linkage(squareform(D, checks=False), method="average")
    g_raw = fcluster(Z_lk, TWIN["thresh"], criterion="distance")
    next_id = int(g_raw.max()) + 1
    groups = g_raw.copy()
    for cid in np.unique(g_raw):
        idx = np.where(g_raw == cid)[0]
        if len(idx) > TWIN["max_size"]:
            subD = D[np.ix_(idx, idx)]
            subZ = linkage(squareform(subD, checks=False), method="average")
            for th in np.linspace(TWIN["thresh"] - 0.005, 0.05, 165):
                sg = fcluster(subZ, th, criterion="distance")
                if np.bincount(sg).max() <= TWIN["max_size"]:
                    for sc in np.unique(sg):
                        groups[idx[sg == sc]] = next_id; next_id += 1
                    break
            else:
                sg = fcluster(subZ, int(np.ceil(len(idx) / TWIN["max_size"])), criterion="maxclust")
                for sc in np.unique(sg):
                    groups[idx[sg == sc]] = next_id; next_id += 1
    del D, Z_lk
else:
    groups = np.ones(len(test), dtype=np.int32)
    S_mat = np.eye(len(test), dtype=np.float32)

size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
eff_sizes = np.where(twin_rows, size, 1)
twin_gate = compute_normalized_twin_gate(eff_sizes, kappa=TWIN_KAPPA)[:, None]
alpha_large = np.clip((eff_sizes.astype(np.float64) - 2.0) / 5.0, 0.0, 1.0)[:, None]
print("topic clusters (size-capped <=18):", (size == 1).sum(), "alone,", twin_rows.sum(), "in clusters of 2..18", flush=True)


In [ ]:
# ==============================================================================
# Stage-2 Host-Exact & Clean Proxy-Matched Sparse Anchors + 120D Forensic Scouts
# ==============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier

SMALL_WORDS_EXT = SMALL_WORDS_C2 | set("however moreover furthermore therefore thus instead rather overall finally first second".split())

def unicode_skeleton_text(t):
    t = re.sub(r"[A-Z][a-z]+", "Tt", t)
    t = re.sub(r"[A-Z]{2,}", "UU", t)
    t = re.sub(r"[a-z]+", "w", t)
    t = re.sub(r"\d+", "0", t)
    return t

def host_distortion_text(t):
    toks = re.findall(r"[A-Za-z]+(?:['\u2019][A-Za-z]+)*|\d+|[^\w\s]|\n", t)
    out = []
    for tok in toks:
        if tok == "\n":
            out.append("NL")
        elif not tok[0].isalnum():
            out.append(f"P_{ord(tok[0])}")
        elif tok[0].isdigit():
            out.append("NUM")
        else:
            has_curly = "\u2019" in tok
            has_str = "'" in tok
            low = tok.lower().replace("\u2019", "'")
            if low in SMALL_WORDS_EXT or "'" in low:
                tag = low + ("_CURLY" if has_curly else ("_STR" if has_str else ""))
                if tok[0].isupper():
                    tag = "CAP_" + tag
                out.append(tag)
            else:
                L = len(tok)
                b = "S" if L <= 4 else ("M" if L <= 7 else "L")
                out.append(("C" if tok[0].isupper() else "c") + b)
    return " ".join(out)

def build_host_exact_views(texts, vecs=None):
    specs = [
        ("skel_char", unicode_skeleton_text, "char", (1, 5), 65000, r"(?u)\b\w\w+\b", 1.0),
        ("distort_tok", host_distortion_text, "word", (1, 3), 90000, r"\S+", 0.9),
        ("char_wb", lambda s: s, "char_wb", (2, 5), 110000, r"(?u)\b\w\w+\b", 0.75),
        ("word_punct", lambda s: s, "word", (1, 2), 80000, r"\w+|[^\w\s]", 0.65),
    ]
    mats, fitted = [], []
    for i, (_, fn, analyzer, ngr, mf, pat, w) in enumerate(specs):
        proc = [fn(t) for t in texts]
        if vecs is None:
            kw = dict(analyzer=analyzer, ngram_range=ngr, max_features=mf, sublinear_tf=True, lowercase=False)
            if analyzer == "word":
                kw["token_pattern"] = pat
            v = TfidfVectorizer(**kw)
            m = v.fit_transform(proc) * w
            fitted.append(v)
        else:
            v = vecs[i]
            m = v.transform(proc) * w
        mats.append(m)
    return hstack(mats, format="csr"), (fitted if vecs is None else vecs)

def build_c2_views_fresh(texts, vecs=None):
    mats, fitted = [], []
    for i, (view, lo, hi, mf, w) in enumerate(VIEWS):
        prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
        proc = [prep(t) for t in texts]
        if vecs is None:
            v = TfidfVectorizer(analyzer=("char" if view in ("char", "shape") else "word"), ngram_range=(lo, hi), max_features=mf, sublinear_tf=True)
            m = v.fit_transform(proc) * w
            fitted.append(v)
        else:
            v = vecs[i]
            m = v.transform(proc) * w
        mats.append(m)
    return hstack(mats, format="csr"), (fitted if vecs is None else vecs)

tr_texts_list = train["text"].tolist()
te_texts_list = test["text"].tolist()
b_norm = np.sqrt(17.0 / 18.0)

# 1. Host-Exact Anchor (100% pure Kaggle host register + Bessel Multiplicity Gate)
X_he_tr, vecs_he = build_host_exact_views(tr_texts_list, vecs=None)
X_he_te, _ = build_host_exact_views(te_texts_list, vecs=vecs_he)
svc_he = LinearSVC(C=0.25, max_iter=2000, random_state=42).fit(X_he_tr, y)
m_he = svc_he.decision_function(X_he_te).astype(np.float32)
m_he_comb = m_he + 0.45 * twin_gate * (bessel_group_center(m_he, groups, twin_rows) * b_norm)
vote("svc_host_exact", softmax(m_he_comb / T_SVC), W_HOST_SVC)
del X_he_tr, X_he_te, vecs_he, svc_he

# 2. Clean Proxy-Matched 8-View Anchor (`passages.csv` funnel-verified + `train.csv` host upweighted)
pa_hits = sorted(glob.glob(f"{INPUT_DIR}/**/passages.csv", recursive=True))
if pa_hits and W_PROXY_SVC > 0:
    pa = pd.read_csv(pa_hits[0])
    pa = pa[
        (pa["passes_funnel"] == True)
        & (pa["is_refusal"] == False)
        & (pa["is_empty"] == False)
        & (pa["split"] != "pilot_range")
        & (~((pa["split"] == "new") & (pa["model"] == "kimi-k3")))
        & (pa["model"].isin(L2I))
    ].copy()
    pa["text"] = pa["text"].fillna("").astype(str)
    px_texts = tr_texts_list + pa["text"].tolist()
    px_y = np.concatenate([y, pa["model"].map(L2I).values])
    px_w = np.concatenate([np.full(len(train), 2.5, dtype=np.float32), np.ones(len(pa), dtype=np.float32)])

    X_px1_tr, vp1 = build_c2_views_fresh(px_texts)
    X_px1_te, _ = build_c2_views_fresh(te_texts_list, vp1)
    X_px2_tr, vp2 = build_host_exact_views(px_texts)
    X_px2_te, _ = build_host_exact_views(te_texts_list, vp2)
    svc_px = LinearSVC(C=0.35, max_iter=2000, random_state=42).fit(
        hstack([X_px1_tr, X_px2_tr * 0.7], format="csr"), px_y, sample_weight=px_w
    )
    m_px = svc_px.decision_function(hstack([X_px1_te, X_px2_te * 0.7], format="csr")).astype(np.float32)
    m_px_comb = m_px + 0.55 * twin_gate * (bessel_group_center(m_px, groups, twin_rows) * b_norm)
    vote("svc_proxy_clean_8view", softmax(m_px_comb / T_SVC), W_PROXY_SVC)
    del pa, px_texts, X_px1_tr, X_px1_te, X_px2_tr, X_px2_te, vp1, vp2, svc_px

# 3. Compass-1 120D Forensic Frontier Scouts (Multi-Regime Augmented HGB + Pre-Saved LR)
F_120_train = extract_forensic_compass1_120d(tr_texts_list)
F_120_test = extract_forensic_compass1_120d(te_texts_list)
f120_tr_dual = np.hstack([F_120_train, bessel_group_center(F_120_train, train["prompt_id"].astype(str).values)])
f120_tr_solo = np.hstack([F_120_train, np.zeros_like(F_120_train)])
f120_tr_aug = np.vstack([f120_tr_dual, f120_tr_dual, f120_tr_solo])
y_f120_aug = np.concatenate([y, y, y])

gate_f25 = compute_normalized_twin_gate(eff_sizes, kappa=2.5).astype(np.float32)[:, None]
f120_te_dual = np.hstack([F_120_test, bessel_group_center(F_120_test, groups, twin_rows) * gate_f25])

f120_te_norm = np.clip(
    (f120_te_dual - apex_st["f120_dual.mean"].float().numpy()) / apex_st["f120_dual.scale"].float().numpy(),
    -6.0, 6.0,
)
logits_f120_lr = f120_te_norm @ apex_st["f120_lr.weight"].float().numpy().T + apex_st["f120_lr.bias"].float().numpy()
hgb_f120 = HistGradientBoostingClassifier(
    max_iter=150, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=5.0, random_state=42
).fit(f120_tr_aug, y_f120_aug)
probs_f120 = 0.45 * softmax(logits_f120_lr / 2.5) + 0.55 * hgb_f120.predict_proba(f120_te_dual)
vote("compass1_forensic_120d", probs_f120, W_FORENSIC120)
del F_120_train, f120_tr_dual, f120_tr_solo, f120_tr_aug, hgb_f120
gc.collect()


In [ ]:
# ==============================================================================
# Bidirectional Structural Anchor: DeBERTa v3 large (Strict VRAM Cleanup)
# ==============================================================================
from transformers import AutoModel, AutoTokenizer

ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from checkpoint head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W_deb, b_deb = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(te_texts_list, truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
reps = [None] * len(ids)
with torch.no_grad():
    for i in range(0, len(ids), 32):
        b_idx = order[i:i + 32]
        b_tok = etok.pad({"input_ids": [ids[j] for j in b_idx]}, padding=True, return_tensors="pt").to("cuda:0")
        h = enc(**b_tok).last_hidden_state
        m = b_tok["attention_mask"][..., None]
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        for j, r in zip(b_idx, pooled.cpu().numpy()):
            reps[j] = r
        del b_tok, h, m, pooled
vote("deberta_v3_large", softmax(np.stack(reps) @ W_deb.T + b_deb), W_DEBERTA)
del enc, head, reps, W_deb, b_deb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ==============================================================================
# Dual-Titan Zero-Leak MoS Layer-Streaming + Dual-Scale Bessel Twin Readout
# (+ Auto-Plug-In for Titan 4 Qwen3-Embedding 8-Depth MoS Scar Experts)
# ==============================================================================
mos_devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else ["cpu"]
f120_scaled_half = np.clip(
    (F_120_test - apex_st["f120_scaler.mean"].float().numpy()) / apex_st["f120_scaler.scale"].float().numpy(),
    -6.0, 6.0,
).astype(np.float32) * 0.5

def apply_dual_scale_titan_readout(Z_titan, t_pfx, name_pfx, w_plain, w_twin):
    X_sc = TestTimeSCache(32, 0.15).apply_scache_resonance(np.hstack([Z_titan, f120_scaled_half]), groups, twin_rows)
    X_p = (X_sc - apex_st[f"{t_pfx}.scaler_plain.mean"].float().numpy()) / apex_st[f"{t_pfx}.scaler_plain.scale"].float().numpy()
    logits_p = (X_p @ apex_st[f"{t_pfx}.plain.weight"].float().numpy().T + apex_st[f"{t_pfx}.plain.bias"].float().numpy()) / T_TITAN
    vote(f"{name_pfx}_plain", softmax(logits_p), w_plain)

    if twin_rows.any():
        X_bc = bessel_group_center(X_sc, groups, twin_rows)
        X_c18 = (X_bc - apex_st[f"{t_pfx}.scaler_centered.mean"].float().numpy()) / apex_st[f"{t_pfx}.scaler_centered.scale"].float().numpy()
        lg_c18 = (X_c18 @ apex_st[f"{t_pfx}.centered.weight"].float().numpy().T + apex_st[f"{t_pfx}.centered.bias"].float().numpy()) / T_TITAN

        X_ca = (X_bc - apex_st[f"{t_pfx}.scaler_centered_arena.mean"].float().numpy()) / apex_st[f"{t_pfx}.scaler_centered_arena.scale"].float().numpy()
        lg_ca = (X_ca @ apex_st[f"{t_pfx}.centered_arena.weight"].float().numpy().T + apex_st[f"{t_pfx}.centered_arena.bias"].float().numpy()) / T_TITAN

        P_twin = (1.0 - alpha_large) * softmax(lg_ca) + alpha_large * softmax(lg_c18)
        vote(f"{name_pfx}_dual_scale_twin", P_twin[twin_rows], w_twin * twin_gate[twin_rows], rows=twin_rows)

# 1. Titan 2: Yi-34B Golden-Slice L30 (30 Layers, 7,168D + 120D = 7,288D)
yi_backbone_dir = os.path.dirname(find("model-00001-of-00002.safetensors"))
tok_yi = AutoTokenizer.from_pretrained(yi_backbone_dir, use_fast=True)
pad_id_yi = tok_yi.pad_token_id if tok_yi.pad_token_id is not None else tok_yi.eos_token_id
raw_ids_yi = tok_yi(te_texts_list, truncation=True, max_length=512, padding=False)["input_ids"]

Z_yi = extract_mos_nf4_streaming(
    yi_backbone_dir, raw_ids_yi, pad_id_yi, mos_devices,
    padding_side=tok_yi.padding_side, tag="Titan2-Yi34B",
    micro_bsz=8, max_tokens_per_dev=60000,
)
assert np.isfinite(Z_yi).all(), "Non-finite features in Z_yi"
apply_dual_scale_titan_readout(Z_yi, "t2", "titan2_yi34b_120d", W_T2_PLAIN, W_T2_TWIN)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. Titan 3: Mistral Large 2 (123B) Golden-Slice L44 (44 Layers, 12,288D + 120D = 12,408D)
mistral_backbone_dir = os.path.dirname(find("model-00001-of-00013.safetensors"))
tok_mistral = AutoTokenizer.from_pretrained(mistral_backbone_dir, use_fast=True)
pad_id_m = tok_mistral.pad_token_id if tok_mistral.pad_token_id is not None else tok_mistral.eos_token_id
raw_ids_m = tok_mistral(te_texts_list, truncation=True, max_length=512, padding=False)["input_ids"]

Z_mistral = extract_mos_nf4_streaming(
    mistral_backbone_dir, raw_ids_m, pad_id_m, mos_devices,
    padding_side=tok_mistral.padding_side, tag="Titan3-Mistral123B",
    micro_bsz=8, max_tokens_per_dev=60000,
)
assert np.isfinite(Z_mistral).all(), "Non-finite features in Z_mistral"
apply_dual_scale_titan_readout(Z_mistral, "t3", "titan3_mistral123b_120d", W_T3_PLAIN, W_T3_TWIN)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 3. Optional Auto-Detected Titan 4: Qwen3-Embedding 8-Depth 5-Fold MoS Scar Experts (Milestone 2)
t4_scars = sorted(glob.glob(f"{INPUT_DIR}/**/titan4_qwen3_emb_8b_fold*_scars.safetensors", recursive=True))
t4_metas = sorted(glob.glob(f"{INPUT_DIR}/**/meta_titan4_qwen3_emb_8b.json", recursive=True))
if t4_scars and t4_metas:
    t4_meta = json.loads(Path(t4_metas[0]).read_text())
    t4_bb_hits = [
        p for p in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True))
        if any(k in p.lower() for k in ("qwen-3-embedding", "qwen3-embedding", "qwen3emb", "qwen3_emb", "titan4"))
    ]
    if t4_bb_hits:
        t4_bb_dir = os.path.dirname(t4_bb_hits[0])
        tok_t4 = AutoTokenizer.from_pretrained(t4_bb_dir, use_fast=True)
        pad_t4 = tok_t4.pad_token_id if tok_t4.pad_token_id is not None else tok_t4.eos_token_id
        win_ids, doc_idx, win_wts = build_covering_windows(tok_t4(te_texts_list, truncation=False, padding=False)["input_ids"], max_len=512, stride=384)
        cap_l = t4_meta["capture_layers_1b"]
        n_dp = len(cap_l)
        t4_probs_acc = np.zeros((len(test), K), dtype=np.float64)
        for f_path in t4_scars:
            st_f = load_file(f_path)
            Z_w = extract_mos_nf4_streaming(t4_bb_dir, win_ids, pad_t4, mos_devices, padding_side="left", tag=Path(f_path).stem, capture_layers=cap_l, scar_tensors=st_f)
            d_m = Z_w.shape[1] // n_dp
            with torch.no_grad():
                dw = F.softmax(st_f["head.depth_logits"].float(), dim=0) * n_dp
                sls = [F.layer_norm(torch.from_numpy(Z_w[:, si*d_m:(si+1)*d_m]), (d_m,), st_f[f"head.depth_norms.{si}.weight"].float(), st_f[f"head.depth_norms.{si}.bias"].float()) * dw[si] for si in range(n_dp)]
                h0 = F.linear(torch.cat(sls, dim=-1), st_f["head.proj.0.weight"].float(), st_f["head.proj.0.bias"].float())
                zw_proj = F.silu(F.layer_norm(h0, (h0.shape[-1],), st_f["head.proj.1.weight"].float(), st_f["head.proj.1.bias"].float())).numpy()
                z_doc = torch.from_numpy(aggregate_window_embeddings(zw_proj, doc_idx, win_wts, len(test), unit_rms=False))
                s_k = F.linear(z_doc, st_f["head.scache_key.weight"].float())
                z_sc = z_doc.clone()
                z_ct = torch.zeros_like(z_doc)
                t_gate = torch.zeros((len(test), 1), dtype=torch.float32)
                for g_id in (np.unique(groups[twin_rows]) if twin_rows.any() else []):
                    m_g = torch.from_numpy(groups == g_id)
                    cnt = int(m_g.sum().item())
                    if cnt > 1:
                        bessel = math.sqrt(cnt / float(cnt - 1))
                        gm = (cnt - 1.0) / (cnt - 1.0 + 3.0)
                        dz = (s_k[m_g] - s_k[m_g].mean(0, keepdim=True)) * bessel
                        w_s = (dz.T @ dz) / float(cnt)
                        z_sc[m_g] = z_doc[m_g] + (float(st_f["head.scache_scale"].item()) * gm) * ((dz @ w_s.T) @ st_f["head.scache_key.weight"].float())
                        t_gate[m_g, 0] = 0.55 * gm
                for g_id in (np.unique(groups[twin_rows]) if twin_rows.any() else []):
                    m_g = torch.from_numpy(groups == g_id)
                    cnt = int(m_g.sum().item())
                    if cnt > 1:
                        bessel = math.sqrt(cnt / float(cnt - 1))
                        z_ct[m_g] = (z_sc[m_g] - z_sc[m_g].mean(0, keepdim=True)) * bessel
                lp = F.linear(F.layer_norm(z_sc, (z_sc.shape[-1],), st_f["head.plain_norm.weight"].float(), st_f["head.plain_norm.bias"].float()), st_f["head.plain_head.weight"].float(), st_f["head.plain_head.bias"].float())
                lc = F.linear(F.layer_norm(z_ct, (z_ct.shape[-1],), st_f["head.twin_norm.weight"].float(), st_f["head.twin_norm.bias"].float()), st_f["head.twin_head.weight"].float(), st_f["head.twin_head.bias"].float())
                t4_probs_acc += ((1.0 - t_gate) * F.softmax(lp, dim=-1) + t_gate * F.softmax(lc, dim=-1)).numpy() / len(t4_scars)
            del st_f, Z_w
        vote("titan4_qwen3emb_8depth_mos_scars", t4_probs_acc, 1.80)


In [ ]:
# ==============================================================================
# System One Collective Inference:
# UniversalS1ReflexiveMLP (625D) + SCache O(1) Woodbury/Cavity Bayesian Head + Cavity-EM + S2
# ==============================================================================
from scipy.special import logsumexp

# 1. Project [Titan2, Titan3, Forensic120] onto 192D SCache Subspace
a_t2 = ((Z_yi - apex_st["scb.sc_t2.mean"].float().numpy()) / apex_st["scb.sc_t2.scale"].float().numpy()) / np.sqrt(Z_yi.shape[1])
b_t3 = ((Z_mistral - apex_st["scb.sc_t3.mean"].float().numpy()) / apex_st["scb.sc_t3.scale"].float().numpy()) / np.sqrt(Z_mistral.shape[1])
c_f = ((F_120_test - apex_st["scb.sc_f.mean"].float().numpy()) / apex_st["scb.sc_f.scale"].float().numpy()) / np.sqrt(F_120_test.shape[1]) * 0.85
X_cat_scb = np.hstack([a_t2, b_t3, c_f]).astype(np.float32)
Z_pca = ((X_cat_scb - apex_st["scb.pca.mean"].float().numpy()) @ apex_st["scb.pca.components"].float().numpy().T).astype(np.float32)
del Z_yi, Z_mistral, a_t2, b_t3, c_f, X_cat_scb
gc.collect()

# 2. UniversalS1ReflexiveMLP 3-Seed Multi-Regime Ensemble (625D -> 320D Residual LayerNorm SiLU -> 18)
X_s1 = np.hstack([
    Z_pca,
    bessel_group_center(Z_pca, groups, twin_rows) * gate_f25,
    F_120_test,
    bessel_group_center(F_120_test, groups, twin_rows) * gate_f25,
    gate_f25,
]).astype(np.float32)

with torch.no_grad():
    p_s1_acc = np.zeros((len(test), K), dtype=np.float64)
    for s_idx in range(3):
        pfx = f"s1.mlp{s_idx}."
        sc_m = apex_st[pfx + "scaler.mean"].float().numpy()
        sc_s = apex_st[pfx + "scaler.scale"].float().numpy()
        xt_s1 = torch.from_numpy(np.clip((X_s1 - sc_m) / sc_s, -6.0, 6.0).astype(np.float32))
        h = F.layer_norm(xt_s1, (xt_s1.shape[-1],), apex_st[pfx + "front_ln.weight"].float(), apex_st[pfx + "front_ln.bias"].float())
        h = F.linear(h, apex_st[pfx + "fc_in.weight"].float(), apex_st[pfx + "fc_in.bias"].float())
        h = F.silu(F.layer_norm(h, (h.shape[-1],), apex_st[pfx + "ln1.weight"].float(), apex_st[pfx + "ln1.bias"].float()))
        hr = F.linear(h, apex_st[pfx + "fc_res.weight"].float(), apex_st[pfx + "fc_res.bias"].float())
        hr = F.silu(F.layer_norm(hr, (hr.shape[-1],), apex_st[pfx + "ln2.weight"].float(), apex_st[pfx + "ln2.bias"].float()))
        h = h + 0.5 * hr
        lg = F.linear(h, apex_st[pfx + "policy_head.weight"].float(), apex_st[pfx + "policy_head.bias"].float())
        p_s1_acc += F.softmax(lg / 1.5, dim=-1).numpy() / 3.0
vote("universal_s1_reflex_mlp", p_s1_acc, W_S1_MLP)

# 3. SCache O(1) Exact Pair/Triple Woodbury + Cavity Bayesian Shared-Prompt Head (< 200 KB state!)
Z_d64 = Z_pca.astype(np.float64)
mu_c = apex_st["scb.mu_c"].float().numpy().astype(np.float64)
cov_eps = apex_st["scb.cov_eps"].float().numpy().astype(np.float64)
cov_prompt = apex_st["scb.cov_prompt"].float().numpy().astype(np.float64)
prec_eps = apex_st["scb.prec_eps"].float().numpy().astype(np.float64)
prec_prompt = apex_st["scb.prec_prompt"].float().numpy().astype(np.float64)
scb_temp = 2.2
same_pen = float(np.log(0.15))

prec0 = np.linalg.inv(cov_prompt + cov_eps)
diff0 = Z_d64[:, None, :] - mu_c[None, :, :]
ll0 = -0.5 * np.einsum("nkd,de,nke->nk", diff0, prec0, diff0) / scb_temp
u_all = -0.5 * np.einsum("nkd,de,nke->nk", diff0, prec_eps, diff0) / scb_temp

prior_cur = softmax(LOGIT)
prior_log = np.log(np.clip(prior_cur, 1e-9, 1.0))
prior_log = prior_log - logsumexp(prior_log, axis=1, keepdims=True)

woodbury_L, wiener_ops = {}, {}
def _get_woodbury_L(m):
    if m not in woodbury_L:
        post_cov = np.linalg.inv(prec_prompt + float(m) * prec_eps)
        ev, eu = np.linalg.eigh(prec_eps @ post_cov @ prec_eps)
        woodbury_L[m] = eu * np.sqrt(np.maximum(ev, 0.0) / scb_temp)
    return woodbury_L[m]

def _get_wiener(m_other):
    if m_other not in wiener_ops:
        post_cov = np.linalg.inv(prec_prompt + float(m_other) * prec_eps)
        wiener_ops[m_other] = (post_cov @ prec_eps, np.linalg.inv(cov_eps + post_cov))
    return wiener_ops[m_other]

ll_scb = ll0.copy()
large_groups = []
for g in np.unique(groups):
    idx = np.where(groups == g)[0]
    m = len(idx)
    if m <= 1 or not twin_rows[idx[0]]:
        ll_scb[idx] = ll0[idx]
    elif m == 2:
        i0, i1 = idx[0], idx[1]
        L2 = _get_woodbury_L(2)
        v0, v1 = diff0[i0] @ L2, diff0[i1] @ L2
        s0 = u_all[i0] + 0.5 * np.sum(v0 * v0, axis=1)
        s1 = u_all[i1] + 0.5 * np.sum(v1 * v1, axis=1)
        coup = v0 @ v1.T
        J0 = s0[:, None] + (s1 + 0.75 * prior_log[i1])[None, :] + coup
        np.fill_diagonal(J0, np.diag(J0) + same_pen)
        J1 = (s0 + 0.75 * prior_log[i0])[:, None] + s1[None, :] + coup
        np.fill_diagonal(J1, np.diag(J1) + same_pen)
        ll_scb[i0] = logsumexp(J0, axis=1)
        ll_scb[i1] = logsumexp(J1, axis=0)
    elif m == 3:
        i0, i1, i2 = idx[0], idx[1], idx[2]
        L3 = _get_woodbury_L(3)
        v0 = diff0[i0] @ L3
        v1 = diff0[i1] @ L3
        v2 = diff0[i2] @ L3
        s0 = u_all[i0] + 0.5 * np.sum(v0 * v0, axis=1)
        s1 = u_all[i1] + 0.5 * np.sum(v1 * v1, axis=1)
        s2 = u_all[i2] + 0.5 * np.sum(v2 * v2, axis=1)
        c01, c02, c12 = v0 @ v1.T, v0 @ v2.T, v1 @ v2.T
        ep = np.eye(K, dtype=np.float64) * same_pen
        b3 = c01[:, :, None] + c02[:, None, :] + c12[None, :, :] + ep[:, :, None] + ep[:, None, :] + ep[None, :, :]
        t0 = b3 + s0[:, None, None] + (s1 + 0.75 * prior_log[i1])[None, :, None] + (s2 + 0.75 * prior_log[i2])[None, None, :]
        t1 = b3 + (s0 + 0.75 * prior_log[i0])[:, None, None] + s1[None, :, None] + (s2 + 0.75 * prior_log[i2])[None, None, :]
        t2 = b3 + (s0 + 0.75 * prior_log[i0])[:, None, None] + (s1 + 0.75 * prior_log[i1])[None, :, None] + s2[None, None, :]
        ll_scb[i0] = logsumexp(t0, axis=(1, 2))
        ll_scb[i1] = logsumexp(t1, axis=(0, 2))
        ll_scb[i2] = logsumexp(t2, axis=(0, 1))
    else:
        large_groups.append(idx)

if large_groups:
    Q_scb = 0.65 * prior_cur + 0.35 * softmax(ll0)
    for _ in range(3):
        res_scb = Z_d64 - (Q_scb @ mu_c)
        for idx in large_groups:
            W_eta, prec_m = _get_wiener(len(idx) - 1)
            cav_sum = res_scb[idx].sum(axis=0, keepdims=True) - res_scb[idx]
            z_clean = Z_d64[idx] - (cav_sum @ W_eta.T)
            df_g = z_clean[:, None, :] - mu_c[None, :, :]
            ll_scb[idx] = -0.5 * np.einsum("mkd,de,mke->mk", df_g, prec_m, df_g) / scb_temp
        Q_scb = 0.50 * prior_cur + 0.50 * softmax(ll_scb)

ll_scb = ll_scb - logsumexp(ll_scb, axis=1, keepdims=True)
vote("scache_o1_woodbury_bayesian_shared_prompt", softmax(ll_scb), W_SCACHE_BAYES)

# 4. Similarity-Weighted Cavity-EM Sibling Repulsion
assert np.isfinite(LOGIT).all(), "Nonfinite ensemble logits"
if len(test) >= 2 and CAVITY_GAMMA > 1e-6:
    P0_log = LOGIT - logsumexp(LOGIT, axis=1, keepdims=True)
    P = np.exp(P0_log)
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        m = len(idx)
        if 2 <= m <= 18:
            sub_S = S_mat[np.ix_(idx, idx)]
            W_sim = np.clip((sub_S - 0.12) / 0.30, 0.0, 1.0)
            np.fill_diagonal(W_sim, 0.0)
            if W_sim.max() > 1e-6:
                W_eff = W_sim / np.maximum(W_sim.sum(axis=1, keepdims=True) / min(float(m - 1), 6.0), 1.0)
                lp_g = P0_log[idx]
                q_g = P[idx].copy()
                for _ in range(5):
                    excl = np.log(np.clip(1.0 - CAVITY_GAMMA * q_g, 1e-4, 1.0))
                    q_g = 0.65 * softmax(lp_g + (W_eff @ excl)) + 0.35 * q_g
                P[idx] = q_g
    P = P / P.sum(axis=1, keepdims=True)
else:
    P = softmax(LOGIT)

# 5. ReflexDecision Escalation -> System 2 Intra-Clan Specialist Verifiers
if S2_BETA > 1e-6:
    sp_sorted = np.sort(P, axis=1)[:, ::-1]
    margins = np.clip(sp_sorted[:, 0] - sp_sorted[:, 1], 0.0, 1.0)
    unc_idx = np.where(margins < 0.35)[0]
    if len(unc_idx) > 0:
        betas = S2_BETA * np.clip(1.0 - margins[unc_idx] / 0.35, 0.0, 1.0)
        X_u = X_s1[unc_idx]
        for clan_name, members in CLANS.items():
            c_mass = P[np.ix_(unc_idx, members)].sum(axis=1, keepdims=True)
            act = c_mass[:, 0] > 0.25
            if act.any():
                sub_u = unc_idx[act]
                b_a = betas[act][:, None]
                cm_a = c_mass[act]
                Xs_c = (X_u[act] - apex_st[f"s2.{clan_name}.mean"].float().numpy()) / apex_st[f"s2.{clan_name}.scale"].float().numpy()
                lg_s2 = (Xs_c @ apex_st[f"s2.{clan_name}.weight"].float().numpy().T + apex_st[f"s2.{clan_name}.bias"].float().numpy()) / 2.0
                if lg_s2.shape[1] == 1:
                    p1 = 1.0 / (1.0 + np.exp(-lg_s2[:, 0]))
                    p_s2 = np.stack([1.0 - p1, p1], axis=1)
                else:
                    p_s2 = softmax(lg_s2)
                p_s1_c = np.clip(P[np.ix_(sub_u, members)] / np.maximum(cm_a, 1e-12), 1e-9, 1.0)
                fused_c = softmax((1.0 - b_a) * np.log(p_s1_c) + b_a * np.log(np.clip(p_s2, 1e-9, 1.0)))
                P[np.ix_(sub_u, members)] = cm_a * fused_c
        P = P / P.sum(axis=1, keepdims=True)

# 6. Size-Scaled Damped Soft Sinkhorn Marginal Calibration
def sinkhorn_damped(P_in, target_mix=0.55, iters=300):
    if target_mix <= 1e-6 or len(P_in) < 50:
        return P_in / P_in.sum(1, keepdims=True)
    P_s = np.clip(P_in, 1e-12, 1.0)
    P_s = P_s / P_s.sum(1, keepdims=True)
    eff_mix = target_mix * min(1.0, len(P_s) / 360.0)
    emp = P_s.sum(0)
    unif = np.full(K, len(P_s) / float(K))
    target = np.maximum((1.0 - eff_mix) * emp + eff_mix * unif, 1e-6)
    c = np.ones(K, dtype=np.float64)
    for _ in range(iters):
        R = P_s * c
        R /= R.sum(1, keepdims=True)
        c *= target / np.maximum(R.sum(0), 1e-12)
    R = P_s * c
    return R / R.sum(1, keepdims=True)

if len(test) >= 50:
    P = sinkhorn_damped(P, target_mix=SINKHORN_MIX, iters=300)

assert np.isfinite(P).all(), "Nonfinite ensemble probability"
assert P.shape == (len(test), K), "Probability shape mismatch"
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"System One Apex V34 complete in {time.time() - T0:.1f}s!", flush=True)
